# How does OLMo 1B Perform on the SAT?

## Importing the Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "allenai/OLMo-1B-hf" # This is the model we are using
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)
model.generation_config.pad_token_id = tokenizer.pad_token_id

### Checkpoint #1
Let's take a step back and think critically about the model we chose to upload. Use these resources to help you answer the following question:

Article on OLMo 1B
https://huggingface.co/allenai/OLMo-1B-hf

Allen AI's Overview of OLMo Models
https://allenai.org/olmo

In [ ]:
# RUN THIS CELL TO ANSWER CHECKPOINT #1
user_response = input("Why do you think we imported OLMo-1B compared to other OLMo models?\n")
file_path = 'answers.txt'
with open(file_path, 'w') as file:
    file.write(user_response)
    file.write('\n') 

In [ ]:
def ask_sat_question(question, options):
    # Construct the full prompt including the question, options, and instructions for the model
    # Note: OLMo-1B is a base model and does not use a chat template, so we prompt it directly
    prompt = f"""Please reason step by step, and put your final answer within \\boxed{{}}.
Here is an SAT-style multiple-choice question:

Question: {question}
Options:
{options}

Please provide your detailed reasoning and select the single best answer.
Answer:"""

    model_inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    # Generate the response
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens = 512, 
        do_sample = True,
        temperature = 0.6,
        top_p = 0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return response

This is an example of how we would ask OLMo to answer a sample question for us. Run it to see how OLMo responds!

> **Note:** Unlike the other models in this series, OLMo 1B is a base (non-instruct) model, meaning it was not fine-tuned to follow instructions. It may be less consistent at following the answer format.

In [ ]:
question_text = "If 3x + 5 = 14, what is the value of x?"
options_text = "A) 2\nB) 3\nC) 4\nD) 5"

response = ask_sat_question(question_text, options_text)
print(response)

### Using the SAT Questionbank

We can use a SAT Questionbank (linked here: https://pinesat.com/api/questions) to pick certain questions based on difficulty or domain. This can let us compare how the model performs depending on the type of "thinking" required.

The difficulty levels are "Easy", "Medium", or "Hard".

The available subjects in the question bank are...

    "Information and Ideas" — reading comprehension, main ideas
    "Standard English Conventions" — grammar, punctuation, sentence structure
    "Expression of Ideas" — rhetoric, author's purpose, transitions
    "Craft and Structure" — literary techniques, vocabulary in context

Take a look at the JSON file to get a sense of how the questionbank is structured: https://pinesat.com/api/questions

### Checkpoint #2

In [ ]:
# RUN THIS CELL TO ANSWER CHECKPOINT #2
user_response = input("Why do you think we are testing the model with these specific domains? Why not math?\n")
file_path = 'answers.txt'
with open(file_path, 'a') as file:
    file.write(user_response)
    file.write('\n') 

First, we will create a function to read the JSON file and fetch a random question for us, depending on the difficulty level and subject area.

In [ ]:
import random, requests

def get_sat_question(difficulty=None, subject=None):
    questions = requests.get("https://pinesat.com/api/questions").json()
    if difficulty: questions = [q for q in questions if q["difficulty"].lower() == difficulty.lower()]
    if subject: questions = [q for q in questions if subject.lower() in q["domain"].lower()]
    return random.choice(questions) if questions else None

Here, we didn't specify a difficulty or domain subject. Take this as an example of how to use this function.

In [ ]:
q = get_sat_question()
question = q["question"]["question"]
choices = q["question"]["choices"]
passage = q["question"]["paragraph"]
answer = q["question"]["correct_answer"]

# Some questions have a passage associated with them for the testtaker to read before answering the question.
# If that's the case, let's use that as the question instead
if passage and passage != "null":
    question = passage

In [ ]:
question

In [ ]:
choices

In [ ]:
passage

In [ ]:
question_text = question
options_text = choices

response = ask_sat_question(question_text, options_text)
print(response)

In [ ]:
answer

### Checkpoint #3

Now, you try! Write some code to pick a random Easy level question and have OLMo attempt the question. Did OLMo get the question right?

In [ ]:
### YOUR CODE HERE

question_text = ...
options_text = ...
response = ask_sat_question(question_text, options_text)
print(response)

In [ ]:
## Did OLMo get the question right? How can you tell? Write the code below.

### YOUR CODE HERE

Let's scale up and create a function to check if OLMo got the right answer automatically. This way, we can run several questions and see how OLMo performs overall.

In [ ]:
import re
def extract_answer(response, correct_answer):
    pattern = r"/correct\sanswer\sis\s(.+)\."
    a = re.search(pattern, correct_answer)
    if correct_answer in response:
        return 'CORRECT'
    else:
        return 'INCORRECT'

In [ ]:
extract_answer(response, answer)

### 12-Question Performance: OLMo 1B

In [ ]:
q1 = get_sat_question('Easy', 'Information and Ideas')
q2 = get_sat_question('Easy', 'Standard English Conventions')
q3 = get_sat_question('Easy', 'Expression of Ideas')
q4 = get_sat_question('Easy', 'Craft and Structure')

q5 = get_sat_question('Medium', 'Information and Ideas')
q6 = get_sat_question('Medium', 'Standard English Conventions')
q7 = get_sat_question('Medium', 'Expression of Ideas')
q8 = get_sat_question('Medium', 'Craft and Structure')

q9 = get_sat_question('Hard', 'Information and Ideas')
q10 = get_sat_question('Hard', 'Standard English Conventions')
q11 = get_sat_question('Hard', 'Expression of Ideas')
q12 = get_sat_question('Hard', 'Craft and Structure')

question_bank = [q1, q2, q3, q4, q5, q6, q7, q8, q9, q10, q11, q12]

In [ ]:
import pandas as pd
df = pd.DataFrame(columns = ['Domain', 'Difficulty', 'Question', 'Choices', 'Response', 'Correct?'])
df

In [ ]:
for q in question_bank:
    question = q["question"]["question"]
    choices = q["question"]["choices"]
    passage = q["question"]["paragraph"]
    answer = q["question"]["correct_answer"]
    domain = q["domain"]
    difficulty = q['difficulty']
    if passage and passage != "null":
        question = passage

    response = ask_sat_question(question, choices)
    check = extract_answer(response, answer)

    new_row = {'Domain': domain, 'Difficulty': difficulty, 'Question': question, 'Choices': choices, 'Response': response, 'Correct?': check}
    df.loc[len(df)] = new_row

In [ ]:
df

In [ ]:
df.to_excel('OLMo1B_results.xlsx')